In [1]:
import sys

from absl import logging
from ferminet.utils import system
from ferminet import base_config
from ferminet import train
from ferminet.configs import atom

# Optional, for also printing training progress to STDOUT.
# If running a script, you can also just use the --alsologtostderr flag.
logging.get_absl_handler().python_handler.stream = sys.stdout
logging.set_verbosity(logging.INFO)


# Define H2 molecule
cfg = base_config.default()
cfg.system.electrons = (1,1)  # (alpha electrons, beta electrons)
cfg.system.molecule = [system.Atom('H', (0, 0, -1)), system.Atom('H', (0, 0, 1))]

# Set training parameters
cfg.batch_size = 4096
cfg.pretrain.iterations = 0
cfg.mcmc.burn_in = 0
cfg.optim.optimizer = 'minsr'


In [2]:
%load_ext autoreload
%autoreload 2

cfg.optim.optimizer = 'minsr'


INFO:absl:Starting QMC with 1 XLA devices per host across 1 hosts.
INFO:absl:No checkpoint found. Training new model.


(667104,)
667104


INFO:absl:Burning in MCMC chain for 0 steps
INFO:absl:Completed burn-in MCMC steps
INFO:absl:Initial energy: 30.6518 E_h


In [20]:
import jax
from ferminet import constants
from ferminet.mcmc import make_mcmc_step
import jax.numpy as jnp

In [ ]:
batch_network = jax.vmap(
      logabs_network, in_axes=(None, 0, 0, 0, 0), out_axes=0
  ) # Multi sample network output

In [ ]:
batch_size_ = 5e5

mcmc_step = make_mcmc_step(
      batch_network,
      batch_size_,
      steps=cfg.mcmc.steps,
      atoms=None,
      blocks=cfg.mcmc.blocks * 0,
  )

In [16]:
mcmc_step_pmapped = constants.pmap(mcmc_step)

In [17]:
data, pmove = mcmc_step_pmapped(
    params, data, sharded_key, mcmc_width
)

In [23]:
def f(p):
    return batch_network(p, data.positions[0], data.spins[0], data.atoms[0], data.charges[0])

In [26]:
data.positions.shape

(1, 4096, 6)

In [19]:
flat_params, unravel_fn = jax.flatten_util.ravel_pytree(params)

In [24]:
jvp_func = lambda x: jax.linearize(f, params)[1](
      unravel_fn(x))

vjp_func = lambda v: jax.flatten_util.ravel_pytree(
    jax.vjp(f, params)[1](v))[0]

def fisher_matmul(
    v, centre_gradients=True, damping=1e-3):
    
    log_psi_jac_v = jvp_func(v)
    update_vector = vjp_func(log_psi_jac_v / batch_size_)
    
    if centre_gradients:
        update_vector -= jnp.mean(update_vector)
    
    update_vector += damping * v
    return update_vector

In [25]:
n_params = flat_params.shape[0]
v = jnp.zeros((n_params))
print(fisher_matmul(v))

TypeError: Cannot concatenate arrays with shapes that differ in dimensions other than the one being concatenated: concatenating along dimension 2 for shapes (4096, 2, 1, 256), (4096, 1, 2, 256), (4096, 1, 2, 256), (4096, 2, 1, 32), (4096, 2, 1, 32).

In [ ]:
evaluate_loss()